# 풀스택 GPT: #4.0~4.6

## Tasks:

- [x] 영화 이름을 가지고 감독, 주요 출연진, 예산, 흥행 수익, 영화의 장르, 간단한 시놉시스 등 영화에 대한 정보로 답장하는 체인을 만드세요.
- [x] LLM은 항상 동일한 형식을 사용하여 응답해야 하며, 이를 위해서는 원하는 출력의 예시를 LLM에 제공해야 합니다.
    > - [x] LLM이 답변 형식을 효과적으로 학습하려면 모든 예시는 동일한 형식을 유지해야 합니다.
- [ ] 예제를 제공하려면 [FewShotPromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples/) 또는 [FewShotChatMessagePromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples_chat/)을 사용하세요.

In [1]:
examples = [
    {
        "title": "The Sentimental Policeman",
        "synopsis": "In this affectionate, leisurely paced comedy, an Odessa policeman is out walking his beat when he discovers an adorable infant abandoned in a cabbage patch. He does his duty and takes the baby to an orphanage, but later he and his wife, who have an unusually affectionate and cozy relationship, decide to try and adopt the little one. What they must go through to accomplish that goal is anything but straightforward.",
        "year": 1992,
        "rate": 3.8,
        "genre": ["Comedy"],
        "director": "Kira Muratova",
        "language": "Russian",
    },
    {
        "title": "Nem Sansão Nem Dalila",
        "synopsis": "Barber's jeep crash against crazy scientist's house, where the latter was building a time-machine. The crash triggers the machine, taking them to Gaza kingdom, circa 1153 B.C., where they get involved in many funny situations. Spoof of Cecil B. DeMille's Samson and Delilah",
        "year": 1955,
        "rate": 6.3,
        "genre": ["Comedy", "Science Fiction",],
        "director": "Carlos Manga",
        "language": "Portuguese",
    },
    {
        "title": "Harmony",
        "synopsis": "Jeong-hye gives a birth to a baby boy in the prison. Her and other inmates create a women's choir to compete in the national choir contest, to meet and greet their families and loving ones.",
        "year": 2010,
        "rate": 7.677,
        "genre": ["Drama", "Music",],
        "director": "Kang Dae-gyu",
        "language": "Korean",
    },
]

In [7]:
import os
from dotenv import load_dotenv
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

load_dotenv()

chat = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME"),
    temperature=0.1,
)

example_prompt = PromptTemplate.from_template(
    """
System: You are a movie geek. You give short info on the movie.
Human: What do you know about the movie, {title}?
AI: Here's what i know.
    Director: {director}
    Casting: "Millie Bobby Brown, Cillian Murphy"
    Rate: {rate}
    Genre: {genre}
    Synopsis: {synopsis}
"""
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="What do you know about the movie, {title}?",
    input_variables=["title"],
)

chain = few_shot_prompt | chat
chain.invoke({"title":"Black Swan"})

AIMessage(content='Here\'s what I know about "Black Swan":\n\n- **Director**: Darren Aronofsky\n- **Casting**: Natalie Portman, Mila Kunis, Vincent Cassel\n- **Rate**: 8.0\n- **Genre**: [\'Drama\', \'Thriller\']\n- **Synopsis**: Nina Sayers, a dedicated and ambitious ballerina, wins the lead role in a New York City production of "Swan Lake." As she strives for perfection, she becomes increasingly consumed by her dual role as the innocent White Swan and the seductive Black Swan. The pressure of competition and her own psychological unraveling lead to a dark and intense journey of obsession, identity, and transformation.')

In [4]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

chat = ChatOpenAI(temperature=0.1)

example_prompt = ChatPromptTemplate.from_messages([
    ("human","What do you know about the movie, {title}?"),
    ("ai", """
     Here's what i know.
     Director: {director}
     Casting: "Millie Bobby Brown, Cillian Murphy"
     Rate: {rate}
     Genre: {genre}
     Synopsis: {synopsis}
    """),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a movie geek. You give short info on the movie."),
    few_shot_prompt,
    ("human","What do you know about the movie, {title}?"),
])

chain = final_prompt | chat
chain.invoke({"title":"Charlie and Chocolate Factory"})

AIMessage(content="\n     Here's what i know.\n     Director: Tim Burton\n     Casting: Johnny Depp, Freddie Highmore\n     Rate: 6.6\n     Genre: ['Adventure', 'Comedy', 'Family']\n     Synopsis: A young boy wins a tour through the most magnificent chocolate factory in the world, led by the world's most unusual candy maker.")